# 03 — Evaluate, export, demo (Day 2)

**Settings:** Accelerator = **GPU T4 x2** (CPU also works, ~10x slower eval), Internet = On.
**Inputs:** dataset `pygpt-python-tokens` **and** *Your Work → Notebook 02 (latest version)* so the
checkpoint mounts under `/kaggle/input/`.

Steps: `eval.py` (metrics + plots + samples, with the GPT-2 baseline) → `export.py` (fp16 ~220 MB)
→ zip a ready-to-upload HF Space bundle → optional: upload to HF Hub / launch a shareable Gradio link.

In [ ]:
# --- Get the code into /kaggle/working/pygpt ---------------------------------
# Option A: public GitHub repo (set REPO_URL). Option B: a Kaggle Dataset named
# "pygpt-code" containing the repo folder (upload the zip -> Datasets -> New Dataset).
REPO_URL = ""            # e.g. "https://github.com/<you>/pygpt.git"
import os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.exists("pygpt"):
    shutil.rmtree("pygpt")
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pygpt"], check=True)
else:
    src = "/kaggle/input/pygpt-code"
    cand = [os.path.join(src, "pygpt"), src]
    for c in cand:
        if os.path.exists(os.path.join(c, "model.py")):
            shutil.copytree(c, "pygpt"); break
    else:
        raise SystemExit("set REPO_URL or add the 'pygpt-code' dataset as input")
print(sorted(os.listdir("pygpt")))

In [ ]:
import glob, os
DATA_DIR = "/kaggle/input/pygpt-python-tokens"
# find ckpt_best.pt from the Notebook 02 input (falls back to ckpt_latest.pt)
cands = sorted(glob.glob("/kaggle/input/*/ckpt_best.pt")) or sorted(glob.glob("/kaggle/input/*/ckpt_latest.pt"))
assert cands, "add Notebook 02's latest version as an input (Add Input -> Your Work)"
CKPT = cands[0]
LOG = os.path.join(os.path.dirname(CKPT), "log.csv")
print("ckpt:", CKPT)
os.chdir("/kaggle/working/pygpt")

In [ ]:
# E1-E5 with the vanilla GPT-2 baseline (~15-25 min on T4, mostly sampling)
!python eval.py --ckpt {CKPT} --data_dir {DATA_DIR} --log_csv {LOG} \
    --out_dir /kaggle/working/results --gpt2

In [ ]:
import json
from IPython.display import Image, display, Markdown
print(json.dumps(json.load(open("/kaggle/working/results/metrics.json")), indent=2))
display(Image("/kaggle/working/results/loss_curve.png"))
display(Markdown(open("/kaggle/working/results/samples.md").read()[:3000]))

In [ ]:
# Export fp16 weights + tokenizer into app/, verify, and zip a complete HF Space bundle
!python export.py --ckpt {CKPT} --out app/model_fp16.pt --tokenizer_src {DATA_DIR}/tokenizer --verify
import shutil, os
os.makedirs("/kaggle/working/space", exist_ok=True)
for f in ("app/app.py", "app/requirements.txt", "app/model_fp16.pt", "model.py", "generate.py", "export.py"):
    shutil.copy(f, "/kaggle/working/space/" + os.path.basename(f))
shutil.copytree("app/tokenizer", "/kaggle/working/space/tokenizer", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/space_bundle", "zip", "/kaggle/working/space")
print("wrote /kaggle/working/space_bundle.zip - upload its contents to a Gradio Space (or use the cell below)")

In [ ]:
# Optional: push weights + the Space in one go. Needs a write token in Kaggle Secrets as HF_TOKEN.
HF_USER = ""                       # e.g. "rehan"; leave empty to skip
if HF_USER:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(f"{HF_USER}/pygpt", exist_ok=True)                                   # model repo
    api.upload_folder(folder_path="/kaggle/working/space", repo_id=f"{HF_USER}/pygpt")
    api.create_repo(f"{HF_USER}/pygpt-demo", exist_ok=True, repo_type="space", space_sdk="gradio")
    api.upload_folder(folder_path="/kaggle/working/space", repo_id=f"{HF_USER}/pygpt-demo", repo_type="space")
    print(f"https://huggingface.co/spaces/{HF_USER}/pygpt-demo")

In [ ]:
# Optional: temporary public Gradio link straight from Kaggle (fallback 2 in PROJECT.md 4.8)
# !cd /kaggle/working/pygpt && GRADIO_SHARE=1 python app/app.py

### Done when
`results/metrics.json` numbers are on the results slide, the Space answers Prompt A in < 15 s,
and `space_bundle.zip` is downloaded as the laptop fallback.